# Task 1. Dataset and Dependencies

### 1.1. Install and import all required dependencies

In [1]:
%pip install -q pgserver==0.1.4 psycopg2-binary sqlalchemy pandas kagglehub

In [2]:
import pandas as pd
import pgserver
import kagglehub

from pathlib import Path
from sqlalchemy import create_engine, text
from IPython.display import display

Start and check server

In [3]:
pg = pgserver.get_server(
    "/tmp/hw5_pg",
    cleanup_mode="stop"
)

engine = create_engine(pg.get_uri(), future=True)

with engine.connect() as conn:
    version = conn.execute(text("SELECT version()")).scalar_one()
    print(version)

PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


Add helper function to execute SQL without expected output to show (e.g. create table)

In [4]:
def run_sql(query):
    with engine.begin() as conn:
        conn.exec_driver_sql(query)

Add helper function to execute SQL queries and show output (e.g., SELECT query)

In [5]:
def run_query(query, params=None):
    with engine.connect() as conn:
        return pd.read_sql_query(
            text(query),
            conn,
            params=params
        )

### 1.2. Download olist dataset

In [6]:
dataset_dir = Path(
    kagglehub.dataset_download("olistbr/brazilian-ecommerce")
)

print("Dataset location:", dataset_dir)

for path in sorted(dataset_dir.glob("*.csv")):
    print(path.name, f"{path.stat().st_size / 1024**2:.2f} MB")

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Dataset location: /kaggle/input/brazilian-ecommerce
olist_customers_dataset.csv 8.62 MB
olist_geolocation_dataset.csv 58.44 MB
olist_order_items_dataset.csv 14.72 MB
olist_order_payments_dataset.csv 5.51 MB
olist_order_reviews_dataset.csv 13.78 MB
olist_orders_dataset.csv 16.84 MB
olist_products_dataset.csv 2.27 MB
olist_sellers_dataset.csv 0.17 MB
product_category_name_translation.csv 0.00 MB


### 1.3. Raw Data Loading

Raw values are loaded as text. Empty CSV fields become SQL NULL; type conversion happens in the typed layer

In [7]:
files = {
    "olist_customers_raw": "olist_customers_dataset.csv",
    "olist_orders_raw": "olist_orders_dataset.csv",
    "olist_order_items_raw": "olist_order_items_dataset.csv",
    "olist_products_raw": "olist_products_dataset.csv",
    "olist_sellers_raw": "olist_sellers_dataset.csv",
    "olist_order_reviews_raw": "olist_order_reviews_dataset.csv",
}

for table_name, file_name in files.items():
    path = dataset_dir / file_name

    df = pd.read_csv(
        path,
        dtype=str,
        keep_default_na=False,
        na_values=[""]
    )

    df.to_sql(
        table_name,
        engine,
        if_exists="replace",
        index=False,
        chunksize=10_000
    )

    print(f"{table_name}: {len(df):,} rows, {len(df.columns)} columns")

olist_customers_raw: 99,441 rows, 5 columns
olist_orders_raw: 99,441 rows, 8 columns
olist_order_items_raw: 112,650 rows, 7 columns
olist_products_raw: 32,951 rows, 9 columns
olist_sellers_raw: 3,095 rows, 4 columns
olist_order_reviews_raw: 99,224 rows, 7 columns


Check number of records in each table

In [8]:
query = """
SELECT 'customers_raw' AS table_name, COUNT(*) AS n_rows
FROM olist_customers_raw
UNION ALL
SELECT 'orders_raw', COUNT(*) FROM olist_orders_raw
UNION ALL
SELECT 'order_items_raw', COUNT(*) FROM olist_order_items_raw
UNION ALL
SELECT 'products_raw', COUNT(*) FROM olist_products_raw
UNION ALL
SELECT 'sellers_raw', COUNT(*) FROM olist_sellers_raw
UNION ALL
SELECT 'order_reviews_raw', COUNT(*) FROM olist_order_reviews_raw
ORDER BY table_name;
"""

display(run_query(query))

,table_name,n_rows
0,customers_raw,99441
1,order_items_raw,112650
2,order_reviews_raw,99224
3,orders_raw,99441
4,products_raw,32951
5,sellers_raw,3095


### 1.4. Typed Schema and Data Conversion

Create typed tables using the schema from learning materials

In [9]:
query = """

DROP MATERIALIZED VIEW IF EXISTS mv_olist_customer_features CASCADE;
DROP TABLE IF EXISTS hw5_referrals CASCADE;
DROP TABLE IF EXISTS olist_order_items    CASCADE;
DROP TABLE IF EXISTS olist_orders         CASCADE;
DROP TABLE IF EXISTS olist_customers      CASCADE;
DROP TABLE IF EXISTS olist_products       CASCADE;
DROP TABLE IF EXISTS olist_sellers        CASCADE;

CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);

CREATE INDEX idx_olist_customers_unique_id
    ON olist_customers (customer_unique_id);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);

CREATE INDEX idx_olist_order_items_seller_order
    ON olist_order_items (seller_id, order_id);

"""

run_sql(query)

Transfer data from raw tables to typed tables (using query from learning materials)

In [10]:
query = """

INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT
    product_id,
    product_category_name,
    product_name_lenght::INTEGER,
    product_description_lenght::INTEGER,
    product_photos_qty::INTEGER,
    product_weight_g::INTEGER,
    product_length_cm::INTEGER,
    product_height_cm::INTEGER,
    product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT
    seller_id,
    seller_zip_code_prefix::INTEGER,
    seller_city,
    seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id::INTEGER,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price::NUMERIC(12, 2),
    freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;

"""

run_sql(query)

Compare raw and typed counts

In [11]:
query = """

SELECT 'customers' AS table_name, COUNT(*) AS n_rows
FROM olist_customers
UNION ALL
SELECT 'orders', COUNT(*) FROM olist_orders
UNION ALL
SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL
SELECT 'products', COUNT(*) FROM olist_products
UNION ALL
SELECT 'sellers', COUNT(*) FROM olist_sellers
ORDER BY table_name;

"""

display(run_query(query))

,table_name,n_rows
0,customers,99441
1,order_items,112650
2,orders,99441
3,products,32951
4,sellers,3095


## Task 2. CTE-Chain Feature Engineering Pipeline

The pipeline uses a **stable reference date** based on the latest order in the historical dataset. Each named stage has one responsibility:

- Stage 0: reference date
- Stage 1: one row per order with total value
- Stage 2: RFM metrics per `customer_id`
- Stage 3: first-purchase cohort quarter
- Stage 4: rolling 30-day AOV and latest-order rank
- Stage 5: cancellation rate
- Stage 6: spending by product category
- Stage 7: preferred category by rank

**Granularity note:** HW5 specifically requires a materialized view with a unique index on `customer_id`, so the pipeline follows the assignment's order-specific `customer_id` grain. Unlike `customer_unique_id` from HW4, this does **not** aggregate repeat purchases of the same real buyer. That limitation should be stated when interpreting the features.

In [12]:
feature_pipeline_sql = """
WITH
reference_date AS (
    -- Stage 0: stable historical reference date
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: one row per order, with status and total order value
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)), 0)::NUMERIC(12, 2)
            AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    GROUP BY o.customer_id, o.order_id, o.order_status, o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: recency, frequency, monetary, first-order date
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*)::INTEGER AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: quarter of first purchase
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average, plus most-recent-order rank
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: fraction of canceled orders
    SELECT
        customer_id,
        AVG(CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END)::NUMERIC(6, 4)
            AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: total value per customer and product category
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0))::NUMERIC(12, 2)
            AS category_revenue
    FROM olist_orders AS o
    INNER JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: highest-spending category per customer
    SELECT customer_id, product_category_name, category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT
    c.customer_id,
    c.customer_unique_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)') AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov ON aov.customer_id = c.customer_id AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc ON pc.customer_id = c.customer_id
"""

### 2.1. Run the pipeline and display sample features

In [13]:
query = f"""
SELECT *
FROM ({feature_pipeline_sql}) AS features
ORDER BY monetary DESC NULLS LAST, customer_id
LIMIT 20;
"""

display(run_query(query))

,customer_id,customer_unique_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
0,1617b1357756262bfa56ab541c47bc16,0a0a92112bd4c708ca5fde585afaa872,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0,telefonia_fixa,13664.08
1,ec5b2ba62e574342386871631fafd3fc,763c8b1c9c68a0229c42c9fc6f662b93,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0,telefonia_fixa,7274.88
2,c6e2731c5b391845f6800c97401a43a9,dc4802a71eae9be1dd28f5d788ceb526,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0,utilidades_domesticas,6929.31
3,f48d464a0baaea338cb25f816991ab1f,459bef486812aa25204be022145caa62,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0,pcs,6922.21
4,3fd6777bbce08a352fddd04e4a7cc8f6,ff4159b92c40ebe40454e3e6a7c35ed6,SP,512,1,6726.66,2017-05-24,2017-04-01,6726.66,0.0,artes,6726.66
5,05455dfa7cd02f13d132aa7a6a9729c6,4007669dec559734d6f53e029e360987,MG,328,1,6081.54,2017-11-24,2017-10-01,6081.54,0.0,agro_industria_e_comercio,6081.54
6,df55c14d1476a9a3467f131269c2477f,da122df9eeddfedc1dc1f5349a1a690c,RJ,565,1,4950.34,2017-04-01,2017-04-01,4950.34,0.0,eletroportateis,4950.34
7,e0a2412720e9ea4f26c1ac985f6a7358,5d0a2980b292d049061542014e8960bf,GO,98,1,4809.44,2018-07-12,2018-07-01,4809.44,1.0,cool_stuff,4809.44
8,24bbf5fd2f2e1b359ee7de94defc4a15,eebb5dda148d3893cdaf5b5ca3040ccb,SP,548,1,4764.34,2017-04-18,2017-04-01,4764.34,0.0,eletroportateis,4764.34
9,3d979689f636322c62418b6346b1c6d2,48e1ac109decbb87765a3eade6854098,PB,118,1,4681.78,2018-06-22,2018-04-01,4681.78,0.0,pcs,4681.78


### 2.2. Smoke-test the feature table

There should be exactly one row per `customer_id`, with no negative recency values or invalid cancellation rates.

In [14]:
query = f"""
SELECT
    COUNT(*) AS feature_rows,
    COUNT(DISTINCT customer_id) AS unique_customer_ids,
    COUNT(*) FILTER (WHERE recency_days < 0) AS negative_recency,
    COUNT(*) FILTER (WHERE cancellation_rate < 0 OR cancellation_rate > 1) AS invalid_cancel_rates,
    COUNT(*) FILTER (WHERE cohort_quarter IS NULL) AS missing_cohort
FROM ({feature_pipeline_sql}) AS features;
"""

display(run_query(query))

,feature_rows,unique_customer_ids,negative_recency,invalid_cancel_rates,missing_cohort
0,99441,99441,0,0,0


### 2.3. Implementation notes

- `order_total` is calculated **before** customer-level aggregation, preventing item-level join multiplication.
- `avg_order_value_30d_recent` is a rolling window value taken from the latest order for each customer ID.
- `preferred_category` is ranked using category value **including freight**, following the course example.
- Features include all order statuses; canceled orders count toward RFM monetary/frequency in this educational pipeline. In a real ML pipeline, this business rule would need further definition.
- The `feature_pipeline_sql` string deliberately has no `ORDER BY` or `LIMIT` at the end. It can be reused unchanged in **Task 4 (Materialized View)** and **Task 5 (EXPLAIN)**, avoiding accidental logic differences.


# Task 3. Recursive CTE

### 3.1. Create and Populate Referral Table

Create a synthetic referral chain using ten existing Olist customer IDs

In [15]:
query = """
DROP TABLE IF EXISTS hw5_referrals CASCADE;

CREATE TABLE hw5_referrals (
    referrer_customer_id TEXT NOT NULL
        REFERENCES olist_customers(customer_id),
    referred_customer_id TEXT PRIMARY KEY
        REFERENCES olist_customers(customer_id),
    referred_at TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    CHECK (referrer_customer_id <> referred_customer_id)
);

WITH first_customers AS (
    SELECT customer_id
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 10
),
numbered AS (
    SELECT
        customer_id,
        ROW_NUMBER() OVER (ORDER BY customer_id) AS rn
    FROM first_customers
)
INSERT INTO hw5_referrals (
    referrer_customer_id,
    referred_customer_id
)
SELECT
    a.customer_id,
    b.customer_id
FROM numbered AS a
JOIN numbered AS b
    ON b.rn = a.rn + 1
ON CONFLICT (referred_customer_id) DO NOTHING;
"""

run_sql(query)

display(run_query("""
    SELECT *
    FROM hw5_referrals
    ORDER BY referrer_customer_id;
"""))

,referrer_customer_id,referred_customer_id,referred_at
0,00012a2ce6f8dcda20d059ce98491703,000161a058600d5901f007fab4c27140,2026-11-01 02:03:07.019046+00:00
1,000161a058600d5901f007fab4c27140,0001fd6190edaaf884bcaf3d49edf079,2026-11-01 02:03:07.019046+00:00
2,0001fd6190edaaf884bcaf3d49edf079,0002414f95344307404f0ace7a26f1d5,2026-11-01 02:03:07.019046+00:00
3,0002414f95344307404f0ace7a26f1d5,000379cdec625522490c315e70c7a9fb,2026-11-01 02:03:07.019046+00:00
4,000379cdec625522490c315e70c7a9fb,0004164d20a9e969af783496f3408652,2026-11-01 02:03:07.019046+00:00
5,0004164d20a9e969af783496f3408652,000419c5494106c306a97b5635748086,2026-11-01 02:03:07.019046+00:00
6,000419c5494106c306a97b5635748086,00046a560d407e99b969756e0b10f282,2026-11-01 02:03:07.019046+00:00
7,00046a560d407e99b969756e0b10f282,00050bf6e01e69d5c0fd612f1bcfb69c,2026-11-01 02:03:07.019046+00:00
8,00050bf6e01e69d5c0fd612f1bcfb69c,000598caf2ef4117407665ac33275130,2026-11-01 02:03:07.019046+00:00


### 3.2. Recursive CTE: Referral Chain Traversal

Business Question: What is the referral depth of each customer in the synthetic referral chain, and what path connects them to the root customer?

In [16]:
query = """
WITH RECURSIVE referral_depth AS (

    -- Anchor: customers who start a referral chain
    SELECT
        c.customer_id,
        1 AS depth,
        ARRAY[c.customer_id]::TEXT[] AS path_ids,
        c.customer_id::TEXT AS path_text
    FROM olist_customers AS c
    WHERE EXISTS (
        SELECT 1
        FROM hw5_referrals AS r
        WHERE r.referrer_customer_id = c.customer_id
    )
    AND NOT EXISTS (
        SELECT 1
        FROM hw5_referrals AS r
        WHERE r.referred_customer_id = c.customer_id
    )

    UNION ALL

    -- Recursive: follow referral links
    SELECT
        r.referred_customer_id,
        rd.depth + 1,
        rd.path_ids || r.referred_customer_id,
        rd.path_text || ' -> ' || r.referred_customer_id
    FROM hw5_referrals AS r
    JOIN referral_depth AS rd
        ON rd.customer_id = r.referrer_customer_id
    WHERE NOT (
        r.referred_customer_id = ANY(rd.path_ids)
    )
      AND rd.depth < 50
)
SELECT
    customer_id,
    depth,
    path_text
FROM referral_depth
ORDER BY depth, customer_id;
"""

display(run_query(query))

,customer_id,depth,path_text
0,00012a2ce6f8dcda20d059ce98491703,1,00012a2ce6f8dcda20d059ce98491703
1,000161a058600d5901f007fab4c27140,2,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
2,0001fd6190edaaf884bcaf3d49edf079,3,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
3,0002414f95344307404f0ace7a26f1d5,4,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
4,000379cdec625522490c315e70c7a9fb,5,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
5,0004164d20a9e969af783496f3408652,6,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
6,000419c5494106c306a97b5635748086,7,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
7,00046a560d407e99b969756e0b10f282,8,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
8,00050bf6e01e69d5c0fd612f1bcfb69c,9,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...
9,000598caf2ef4117407665ac33275130,10,00012a2ce6f8dcda20d059ce98491703 -> 000161a058...


Olist does not contain referral relationships, so hw5_referrals is a synthetic table created for this exercise.

The recursive query starts from customers who have referred another customer but were not referred themselves. It then follows the referral links and calculates each customer's depth and path.

The visited-ID array prevents a customer from appearing twice in the same traversal path, while the depth limit prevents excessively long recursion. NOT EXISTS is used for the root lookup instead of NOT IN to avoid potential NULL-related issues.

# Task 4. Materialized View

### 4.1. Create Materialized View

Store the complete customer-level feature dataset in a Materialized View

In [17]:
query = f"""
DROP MATERIALIZED VIEW IF EXISTS mv_olist_customer_features CASCADE;

CREATE MATERIALIZED VIEW mv_olist_customer_features AS
{feature_pipeline_sql.strip().rstrip(';')};
"""

run_sql(query)

### 4.2. Create UNIQUE INDEX

In [18]:
query = """
CREATE UNIQUE INDEX IF NOT EXISTS
    idx_mv_olist_customer_features_customer_id
ON mv_olist_customer_features (customer_id);
"""

run_sql(query)

Check added index

In [19]:
query = """
SELECT
    indexname,
    indexdef
FROM pg_indexes
WHERE schemaname = current_schema()
  AND tablename = 'mv_olist_customer_features';
"""

display(run_query(query))

,indexname,indexdef
0,idx_mv_olist_customer_features_customer_id,CREATE UNIQUE INDEX idx_mv_olist_customer_feat...


### 4.3. Verify Materialized Features

In [20]:
query = """
SELECT
    COUNT(*) AS feature_rows,
    COUNT(DISTINCT customer_id) AS unique_customer_ids,
    COUNT(*) FILTER (WHERE recency_days < 0)
        AS negative_recency,
    COUNT(*) FILTER (
        WHERE cancellation_rate NOT BETWEEN 0 AND 1
    ) AS invalid_cancellation_rates,
    COUNT(*) FILTER (WHERE cohort_quarter IS NULL)
        AS missing_cohort
FROM mv_olist_customer_features;
"""

display(run_query(query))

,feature_rows,unique_customer_ids,negative_recency,invalid_cancellation_rates,missing_cohort
0,99441,99441,0,0,0


Show 5 examples

In [21]:
query = """
SELECT
    customer_id,
    recency_days,
    frequency,
    monetary,
    cohort_quarter,
    avg_order_value_30d_recent,
    cancellation_rate,
    preferred_category
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST, customer_id
LIMIT 5;
"""

display(run_query(query))

,customer_id,recency_days,frequency,monetary,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category
0,1617b1357756262bfa56ab541c47bc16,384,1,13664.08,2017-07-01,13664.08,0.0,telefonia_fixa
1,ec5b2ba62e574342386871631fafd3fc,95,1,7274.88,2018-07-01,7274.88,0.0,telefonia_fixa
2,c6e2731c5b391845f6800c97401a43a9,613,1,6929.31,2017-01-01,6929.31,0.0,utilidades_domesticas
3,f48d464a0baaea338cb25f816991ab1f,85,1,6922.21,2018-07-01,6922.21,0.0,pcs
4,3fd6777bbce08a352fddd04e4a7cc8f6,512,1,6726.66,2017-04-01,6726.66,0.0,artes


### 4.4. REFRESH MATERIALIZED VIEW CONCURRENTLY

In [22]:
query = """
REFRESH MATERIALIZED VIEW CONCURRENTLY
    mv_olist_customer_features;
"""

run_sql(query)

print("Materialized View refreshed successfully.")

Materialized View refreshed successfully.


Check data availability after refresh

In [23]:
query = """
SELECT COUNT(*) AS feature_rows_after_refresh
FROM mv_olist_customer_features;
"""

display(run_query(query))

,feature_rows_after_refresh
0,99441


The Materialized View stores the complete customer feature pipeline from Task 2, including RFM, cohort quarter, rolling AOV, cancellation rate, and preferred category.

The same feature_pipeline_sql is reused to keep feature definitions consistent. A unique index on customer_id ensures one feature row per customer and enables concurrent refresh.

REFRESH MATERIALIZED VIEW CONCURRENTLY recalculates the stored features without blocking concurrent reads. However, the MV is not updated automatically when source tables change. A refresh schedule is needed to balance data freshness and computation cost.

# Task 5. EXPLAIN ANALYZE and Reflection

### 5.1. EXPLAIN: Full CTE Pipeline

Measure the cost of recalculating all customer features from the source tables

In [24]:
query = f"""
EXPLAIN (ANALYZE, BUFFERS)
{feature_pipeline_sql.strip().rstrip(';')};
"""

full_plan_df = run_query(query)
full_plan = full_plan_df.iloc[:, 0].astype(str).tolist()

print("\n".join(full_plan))

Hash Left Join  (cost=81654.35..98271.38 rows=84100 width=186) (actual time=3077.718..3769.488 rows=99441 loops=1)
  Hash Cond: (c.customer_id = ranked.customer_id)
  Buffers: shared hit=12448, temp read=11279 written=11431
  CTE stage1_orders_total
    ->  Finalize GroupAggregate  (cost=16725.66..27526.12 rows=88592 width=124) (actual time=409.391..875.296 rows=99441 loops=1)
          Group Key: o_1.order_id
          Buffers: shared hit=4158, temp read=1545 written=1548
          ->  Gather Merge  (cost=16725.66..25488.91 rows=64913 width=136) (actual time=409.375..691.920 rows=99526 loops=1)
                Workers Planned: 1
                Workers Launched: 1
                Buffers: shared hit=4158, temp read=1545 written=1548
                ->  Partial GroupAggregate  (cost=15725.65..17186.19 rows=64913 width=136) (actual time=363.624..485.021 rows=49763 loops=2)
                      Group Key: o_1.order_id
                      Buffers: shared hit=4158, temp read=1545 writte

### 5.2. EXPLAIN: Materialized View Lookup

Measure the cost of retrieving one customer's cached features using the unique index

In [25]:
target_customer_id = run_query("""
    SELECT MIN(customer_id) AS customer_id
    FROM mv_olist_customer_features;
""").iloc[0, 0]

query = """
EXPLAIN (ANALYZE, BUFFERS)
SELECT *
FROM mv_olist_customer_features
WHERE customer_id = :customer_id;
"""

mv_lookup_df = run_query(
    query,
    params={"customer_id": target_customer_id}
)

mv_lookup_plan = mv_lookup_df.iloc[:, 0].astype(str).tolist()

print("Customer ID:", target_customer_id)
print("\n".join(mv_lookup_plan))

Customer ID: 00012a2ce6f8dcda20d059ce98491703
Index Scan using idx_mv_olist_customer_features_customer_id on mv_olist_customer_features  (cost=0.42..8.44 rows=1 width=186) (actual time=0.023..0.024 rows=1 loops=1)
  Index Cond: (customer_id = '00012a2ce6f8dcda20d059ce98491703'::text)
  Buffers: shared hit=4
Planning Time: 0.066 ms
Execution Time: 0.039 ms


### 5.3. Full MATERIALIZED VIEW Read

In [26]:
query = """
EXPLAIN (ANALYZE, BUFFERS)
SELECT *
FROM mv_olist_customer_features;
"""

mv_full_df = run_query(query)
mv_full_plan = mv_full_df.iloc[:, 0].astype(str).tolist()

print("\n".join(mv_full_plan))

Seq Scan on mv_olist_customer_features  (cost=0.00..2978.41 rows=99441 width=186) (actual time=0.009..12.746 rows=99441 loops=1)
  Buffers: shared hit=1984
Planning Time: 0.066 ms
Execution Time: 17.152 ms


### 5.4. Summary of Execution Times

In [27]:
import re

def execution_time_ms(plan_lines):
    plan_text = "\n".join(plan_lines)
    match = re.search(
        r"Execution Time:\s+([\d.]+)\s+ms",
        plan_text
    )
    return float(match.group(1)) if match else None


results = pd.DataFrame([
    {
        "scenario": "Full CTE pipeline",
        "execution_ms": execution_time_ms(full_plan)
    },
    {
        "scenario": "MV single-customer lookup",
        "execution_ms": execution_time_ms(mv_lookup_plan)
    },
    {
        "scenario": "MV full read",
        "execution_ms": execution_time_ms(mv_full_plan)
    }
])

display(results)

,scenario,execution_ms
0,Full CTE pipeline,3784.041
1,MV single-customer lookup,0.039
2,MV full read,17.152


# Reflection

This homework showed the difference between recalculating customer features and reading cached results from a Materialized View. In my EXPLAIN ANALYZE results, the full CTE pipeline took 3,784.041 ms, compared with 17.152 ms for reading all 99,441 MV rows. A single-customer lookup took 0.039 ms using an Index Scan. This last lookup is not directly comparable with full recomputation because it returns only one row.

The rolling 30-day AOV appears to be one of the most expensive features. Its execution branch includes WindowAgg operations and sorting, while preferred-category ranking requires another window operation and aggregation. The full plan also contains Hash Left Joins, GroupAggregate and HashAggregate nodes. Some sorts used external merge, and customer-level HashAggregate nodes ran in 21 batches with temporary disk usage. These operations explain much of the pipeline cost, although the plan does not provide a completely isolated time for each feature.

I would cache these features in an MV and refresh it daily for an offline ML workflow. More frequent refreshes might be justified for near-real-time recommendations, but would increase recomputation cost. Concurrent refresh allows readers to access the previous snapshot while the MV is being updated. A regular VIEW would be better for small datasets or queries that must use the latest source values.

The synthetic recursive CTE uses a visited-ID array and a maximum depth to terminate safely. Without protection, a cycle such as A to B to A could repeatedly revisit nodes. For this pipeline, low serving latency and controlled recomputation cost are the main priorities. Freshness requirements depend on the application, and extra storage for the MV is an acceptable trade-off. Although the homework uses customer_id, real-buyer ML features would normally use customer_unique_id.